# Команда 12. Смирнова М., Дворяшина И., Шумакова В., Шилкова А.

# Импорты библиотек

In [ ]:
# Стандартные библиотеки Python
import json
import re
import warnings

# Сторонние библиотеки
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

In [ ]:
# Отключаем все локальные предупреждения
warnings.filterwarnings('ignore')

# Чтение данных

In [ ]:

df_path = "videostreaming_platform.csv"

df = pd.read_csv(df_path)

df.head()

In [ ]:
df.info()

In [ ]:
# Проверка наличия пустых значений
print("Пропуски в данных: \n", df.isna().sum())

## Пропуски в данных обнаружены у признака 'city' (308) и 'favourite_genre' (7952)

In [ ]:
df.describe()

In [ ]:
df.describe(include = 'object')

In [ ]:
print("Уникальные значения категориального признака city: ", df['city'].unique())
print("Уникальные значения категориального признака device: ", df['device'].unique())
print("Уникальные значения категориального признака source: ", df['source'].unique())
print("Уникальные значения категориального признака favourite_genre: ", df['favourite_genre'].unique())

# Чистка данных

In [ ]:
# Преобразование дат в datetime
df['start_trial_date'] = pd.to_datetime(df['start_trial_date'])

# Замена NaN на "Unknown"
df['city'] = df['city'].fillna('Unknown')
df['favourite_genre'] = df['favourite_genre'].fillna('Unknown')

# Feature Engineering
# Создание новой колонки день недели начала триала
df['trial_start_day_of_week'] = df['start_trial_date'].dt.dayofweek
df.info()

In [ ]:
# Построение боксплота распределения и аномалии в данных о среднем времени просмотра
plt.figure(figsize=(8, 4))
df["avg_min_watch_daily"].plot(kind="box")
plt.title("Boxplot: avg_min_watch_daily")
plt.show()

In [ ]:
# Расчет процентилей
percentiles = df['avg_min_watch_daily'].quantile([0.5, 0.75, 0.9, 0.95, 0.99, 0.999])
print("Процентили времени просмотра:")
print(percentiles)

Поскольку 80 минут (максимальное значение) - адекватное время просмотра, и в целом для стриминга возможно скошенное распределение, принято решение не проводить очистку от выбросов и работать с исходными данными, чтобы не потерять полезную информацию для анализа конверсии

# Построение диаграмм категориальных признаков для анализа распределение данных

In [ ]:
# Определяем категориальные признаки
categorical_columns = ['city', 'device', 'source', 'favourite_genre', 'churn']

fig, axes = plt.subplots(3, 2, figsize=(16, 15))
axes = axes.flatten()

color_palettes = [
    plt.cm.Set3,
    plt.cm.Pastel1,
    plt.cm.Set2,
    plt.cm.Accent,
    plt.cm.Pastel2
]

for i, col in enumerate(categorical_columns):
    if i < len(axes):
        ax = axes[i]

        # Получаем данные
        value_counts = df[col].value_counts()
        total = len(df[col])

        # Если уникальных значений много, показываем топ-8
        if len(value_counts) > 8:
            top_values = value_counts.head(8)
            others_count = value_counts[8:].sum()
            if others_count > 0:
                top_values['Остальные'] = others_count
            title_suffix = " (топ-8 + другие)"
        else:
            top_values = value_counts
            title_suffix = ""

        n_colors = len(top_values)
        if i < len(color_palettes):
            colors = color_palettes[i](np.linspace(0, 1, max(n_colors, 8)))
        else:
            colors = plt.cm.tab20c(np.linspace(0, 1, n_colors))

        colors = colors[:n_colors]

        wedges, texts, autotexts = ax.pie(top_values.values,
                                          labels=top_values.index.astype(str),
                                          autopct='%1.1f%%',
                                          startangle=90,
                                          colors=colors)

        ax.set_title(f'{col}{title_suffix}\nВсего: {total} записей',
                    fontsize=12, fontweight='bold')

        for autotext in autotexts:
            autotext.set_color('black')
            autotext.set_fontsize(9)
            autotext.set_fontweight('bold')

for i in range(len(categorical_columns), len(axes)):
    axes[i].axis('off')

plt.suptitle('Распределение категориальных признаков', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Все данные выглядят адекватно**

# Построение диаграмм числовых признаков для анализа распределение данных

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Распределение числовых признаков с KDE', fontsize=16, fontweight='bold', y=1.05)

numeric_features = ['number_of_days_logged', 'avg_min_watch_daily']

colors = ['steelblue', 'coral']

for idx, (feature, color) in enumerate(zip(numeric_features, colors)):
    ax = axes[idx]

    sns.histplot(data=df, x=feature, bins=30,
                 kde=True, color=color, edgecolor='black', alpha=0.7, ax=ax)

    mean_val = df[feature].mean()
    median_val = df[feature].median()
    std_val = df[feature].std()

    feature_names = {
        'number_of_days_logged': 'Количество дней активности',
        'avg_min_watch_daily': 'Среднее время просмотра в день (мин)'
    }

    feature_name = feature_names.get(feature, feature)
    ax.set_title(f'Распределение {feature_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel(feature_name, fontsize=12)
    ax.set_ylabel('Плотность', fontsize=12)

    # Добавляем линии для среднего и медианы
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2,
               label=f'Среднее: {mean_val:.1f}')
    ax.axvline(median_val, color='green', linestyle='--', linewidth=2,
               label=f'Медиана: {median_val:.1f}')

    # Добавляем область ±1 стандартное отклонение
    ax.axvspan(mean_val - std_val, mean_val + std_val, alpha=0.2, color='yellow',
               label=f'±1 ст.откл.: [{mean_val-std_val:.1f}, {mean_val+std_val:.1f}]')

    ax.legend(loc='upper right', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

for feature in numeric_features:
    print(f"\n{feature}:\n")

    # Асимметрия и эксцесс
    skewness = df[feature].skew()
    kurtosis = df[feature].kurtosis()

    skewness_interpretation = "симметричное" if abs(skewness) < 0.5 else \
                             "умеренно скошенное" if abs(skewness) < 1 else \
                             "сильно скошенное"

    print(f"Асимметрия: {skewness:.3f} ({skewness_interpretation})")
    print(f"Эксцесс: {kurtosis:.3f}")


**Все данные выглядят адекватно**

# Построение тепловой диаграммы, чтобы найти линейные зависимости между признаками

In [ ]:
# Сначала сделаем one-hot преобразование категориальных признаков
categorical_cols = ['city', 'device', 'source', 'favourite_genre']
data_encoded = pd.get_dummies(df, columns=categorical_cols)

print(data_encoded.head())

In [ ]:
# Удалим колонку 'user_id', чтобы не мешала анализу
del data_encoded['user_id']

In [ ]:
# Строим тепловую диаграмму
correlation_matrix = data_encoded.corr()

plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})

plt.title("Матрица корреляций признаков", fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

Наблюдается **отрицательная корреляция** между **churn** и **avg_min_watch_daily** (-0.47) - чем больше время просмотров в день, тем меньше вероятность ухода

## Гипотеза 0. Какие либо признаки (город, устройство, источник, любимый жанр) связаны с оттоком пользователей.

### Стратегия проверки
Для каждого one-hot столбца создать таблицу сопряженности 2×2 с churn
Рассчитать коэффициент фи (φ) для измерения силы и направления связи
Рассчитать p-value через тест хи-квадрат для проверки значимости

### Статистический метод

Тест хи-квадрат Пирсона для независимости с последующим расчетом коэффициента фи (φ)

#### Обоснование выбора теста:

Природа данных определяет метод:
 - Целевая переменная churn — бинарная (0 или 1, продлил/отписался)
 - Признаки — категориальные, преобразованные в one-hot (город, устройство, источник, жанр)

Хи-квадрат — стандартный тест для категориальных данных:
Проверяет независимость двух категориальных переменных.
Не требует предположений о нормальности распределения.

Коэффициент фи (φ) дополняет хи-квадрат:
Хи-квадрат показывает «есть ли связь» (p-value).
φ показывает «насколько сильна связь» (величина эффекта) и «в какую сторону» (знак).



# Проверка гипотезы статистическими тестами

### H₀: Категориальные признаки (город, устройство, источник, любимый жанр) не связаны с оттоком пользователей. P(churn=1 | X=1) = P(churn=1 | X=0)
### H₁: Существует статистически значимая связь между хотя бы одним категориальным признаком и оттоком. P(churn=1 | X=1) ≠ P(churn=1 | X=0)

In [ ]:
from scipy.stats import chi2_contingency

results = []

for col in data_encoded.columns:
    if any(cat in col for cat in ['city_', 'device_', 'source_', 'favourite_genre_']):
        conf_matrix = pd.crosstab(data_encoded[col], data_encoded['churn'])

        # Фи-коэффициент для 2x2 таблицы
        if conf_matrix.shape == (2, 2):
            a, b, c, d = conf_matrix.values.flatten()
            phi = (a*d - b*c) / np.sqrt((a+b)*(c+d)*(a+c)*(b+d))
        else:
            # V Крамера для таблиц больше 2x2
            chi2 = chi2_contingency(conf_matrix)[0]
            n = conf_matrix.sum().sum()
            phi = np.sqrt(chi2 / n)

        # p-value для проверки значимости
        _, p_value, _, _ = chi2_contingency(conf_matrix)

        results.append({
            'feature': col,
            'phi_coefficient': round(phi, 4),
            'p_value': round(p_value, 6),
            'significant': p_value < 0.05
        })

phi_results = pd.DataFrame(results)
phi_results = phi_results.sort_values('phi_coefficient', key=abs, ascending=False)

print("Связь категориальных признаков с оттоком:")
print(phi_results[['feature', 'phi_coefficient', 'p_value', 'significant']])

### Выводы: Статистически значимых связей категориальных переменных с целевой нет

# Проанализируем как отток меняется в зависимости от среднего времени просмотра в день.

In [ ]:
bins_minutes = [0, 5, 10, 15, 20, 25, 30, 35, 40, 60, 81]
labels = ['<5 мин', '5-10', '10-15', '15-20', '20-25', '25-30', '30-35', '35-40', '40-60', '60-81']

df['watch_category'] = pd.cut(df['avg_min_watch_daily'], 
                               bins=bins_minutes, 
                               labels=labels, 
                               right=False)

df.groupby('watch_category')['churn'].mean().plot(kind='bar', figsize=(10, 6))

plt.title('Отток по времени просмотра')
plt.ylabel('Доля оттока')
plt.xlabel('Среднее время просмотра в день')
plt.xticks(rotation=45)
plt.show()

## Гипотеза 1: Среднее время просмотра влияет на покупку

Чем выше среднее время просмотра в день, тем выше конверсия в подписку.

### Стратегия проверки

Разобьем на две группы:

1. те, кто совершил подписку
2. те, кто нет

Проверим, различаются ли статистически у них среднее время просмотра.

### Статистический метод

Тест Манна-Уитни

In [ ]:
plt.figure(figsize=(8, 6))

df_sorted = df.copy()
df_sorted["churn_str"] = df_sorted["churn"].map({1: "Не подписались", 0: "Подписались"})

sns.violinplot(
    data=df_sorted,
    x="churn_str",
    y="avg_min_watch_daily",
    palette=["#ff9999", "#068615"],
    cut=0
)

plt.title("Распределение времени просмотра по группам подписки")
plt.xlabel("Статус")
plt.ylabel("Среднее время просмотра в день")
plt.grid(axis="y", alpha=0.3)

plt.show()


Взяли Манна-Уитни, так как у нас ненормальное распределение среднего времени просмотра.


H₀ (нулевая гипотеза): медианное (или среднее) время просмотра у подписавшихся не больше, чем у отписавшихся.

H₁ (альтернативная гипотеза): медианное время просмотра у подписавшихся больше, чем у отписавшихся.

In [ ]:
df["converted"] = (df["churn"] == 0).astype(int)
group_buy = df[df["converted"] == 1]["avg_min_watch_daily"]
group_not = df[df["converted"] == 0]["avg_min_watch_daily"]


In [ ]:
x = group_buy.values
y = group_not.values

median_buy = np.median(x)
median_not = np.median(y)
# ранги
all_vals = np.concatenate([x, y])
ranks = np.argsort(np.argsort(all_vals)) + 1

# ранги обратно в группы
r_x = ranks[:len(x)]
r_y = ranks[len(x):]

# U-статистика
U1 = r_x.sum() - len(x)*(len(x)-1)/2
U2 = r_y.sum() - len(y)*(len(y)-1)/2
U = min(U1, U2)

# мат. ожидание и дисперсия
mu = len(x)*len(y)/2
sigma = np.sqrt(len(x)*len(y)*(len(x)+len(y)+1)/12)

# Z-оценка
Z = (U - mu) / sigma

# p-value через нормальное распределение
from math import erf, sqrt
p_value = 2 * (1 - 0.5 * (1 + erf(abs(Z) / sqrt(2))))

print("Результаты теста Манна–Уитни на различие среднего времени просмотра\n")
print(f"Медианное время просмотра (подписались): {median_buy:.2f} мин/день")
print(f"Медианное время просмотра (не подписались): {median_not:.2f} мин/день\n")
print(f"U-статистика: {U:.2f}")
print(f"Z-значение: {Z:.2f}")
print(f"p-value: {p_value:.3g}\n")

if p_value < 0.05:
    print("Вывод: различия статистически значимы (p < 0.05).**")
    if median_buy > median_not:
        print("Пользователи, которые подписались, смотрели существенно больше.")
    else:
        print("Не подписавшиеся смотрели больше (что маловероятно для такой задачи).")
else:
    print("Вывод: статистически значимых различий НЕ обнаружено.")

print("\nИнтерпретация:")
print("Чем больше пользователь смотрит в триал, тем выше вероятность, что он купит подписку.")
print("Этот признак — сильный маркер вовлечённости и должен использоваться в модели.")

In [ ]:
plt.figure(figsize=(5, 7))
sns.boxplot(
    data=df,
    x="churn",
    y="avg_min_watch_daily",
    palette={"0": "green", "1": "pink"}
)

plt.xticks([0, 1], ["Подписались (0)", "Не подписались (1)"])
plt.title("Время просмотра по группам: подписались vs не подписались")
plt.xlabel("Группа churn")
plt.ylabel("Среднее время просмотра")
plt.grid(axis='y')

plt.show()


**Гипотеза 1 подтверждена. Пользователи с более высоким средним временем просмотра значительно чаще покупают подписку.**

Разница по Манна–Уитни: Z = –75.4, p < 0.001

Это означает, что распределения существенно различаются.

Глубина просмотра - важдый маркер конверсии.


In [ ]:
colors = df["churn"].map({0: "green", 1: "pink"})

plt.figure(figsize=(10, 6))

plt.scatter(
    df["avg_min_watch_daily"],
    df["number_of_days_logged"],
    c=colors,
    alpha=0.5,
    s=20,
)

plt.xlabel("Среднее время просмотра (мин в день)")
plt.ylabel("Количество дней логинов")
plt.title("Логины и время просмотра: подписались vs не подписались")

import matplotlib.patches as mpatches
green_patch = mpatches.Patch(color='green', label='Подписались (churn=0)')
pink_patch  = mpatches.Patch(color='pink', label='Не подписались (churn=1)')
plt.legend(handles=[green_patch, pink_patch])

plt.grid(True, alpha=0.3)
plt.show()

Даже если пользователь заходил 1-2 но смотрел больше 10-20 минут вероятность подписку выше, чес у тех кто был в сервисе меньше 10 минут;

С дрегой строны даже 7 дней подключений, не гарантируют подключение платной подписки, там так же есть большое количество отписавшихся среди тех кто смотрел меньше 10 дней.

Посмотрим дальше насколько это достоверно.

### Двухфакторный анализ — метод множественной логистической регрессии
Проверяем, является ли avg_min_watch_daily значимым предиктором после учёта number_of_days_logged.

Логистическая регрессия проверяет значимость коэффициентов.Поэтому нулевая и альтернативная гипотезы следующие:

*Для переменной avg_min_watch_daily:*

H0: avg_min_watch_daily не влияет на отток

H1: avg_min_watch_daily влияет на отток


*Для переменной number_of_days_logged:*

H0: number_of_days_logged не влияет на отток

H1: number_of_days_logged влияет на отток

In [ ]:
df2 = df[["avg_min_watch_daily", "number_of_days_logged", "churn"]].copy()
X = df2[["avg_min_watch_daily", "number_of_days_logged"]]
X = sm.add_constant(X)
y = df2["churn"]

model = sm.Logit(y, X).fit()
print(model.summary())

In [ ]:
coef_watch = model.params["avg_min_watch_daily"]
coef_days = model.params["number_of_days_logged"]
coef_watch

**Посчитаем как влияет дополнительная минута просмотра и дополнительный день входа на конверсию в подписку**


In [ ]:
coef_watch = model.params["avg_min_watch_daily"]
coef_days = model.params["number_of_days_logged"]

p_days = model.pvalues["avg_min_watch_daily"]
p_days = model.pvalues["number_of_days_logged"]


or_watch = np.exp(coef_watch)
or_days = np.exp(coef_days)

effect_watch = (1 - or_watch) * 100
effect_days = (1 - or_days) * 100

print("Интерпретация логистической регрессии")

print(f"Коэффициент avg_min_watch_daily = {coef_watch:.4f}")
print(f"P-value для времени просмотра: {p_days:.4f}")
print(f"Odds ratio = {or_watch:.4f}")
print(f"Каждая дополнительная минута просмотра снижает шансы отписки на {effect_watch:.2f}%\n")

print(f"Коэффициент number_of_days_logged = {coef_days:.4f}")
print(f"P-value для числа дней логинов: {p_days:.4f}")
print(f"Odds ratio = {or_days:.4f}")
print(f"Каждый дополнительный день логина снижает шансы отписки на {effect_days:.2f}%")

Ключевым драйвером покупки подписки является не просто факт захода в сервис, а глубина просмотра контента.

Нужно сосредоточиться на времени удержания клиента, например лучше настроить систему рекомендаций.

Или с другой стороны сделать какую-то модель которая будет предсказывать холодных клиентов которые смотрят до 5-10 минут в день и не тратить на них рекламные бюджеты.

# Проанализируем какое минимальное время просмотра защищает от оттока

In [ ]:
# Анализ пороговых значений для времени просмотра
threshold_analysis = []
for threshold in [5, 10, 15, 30, 60]:
    churn_below = df[df['avg_min_watch_daily'] < threshold]['churn'].mean()
    churn_above = df[df['avg_min_watch_daily'] >= threshold]['churn'].mean()
    threshold_analysis.append({
        'threshold': threshold,
        'churn_below': churn_below,
        'churn_above': churn_above,
        'diff': churn_below - churn_above
    })

threshold_df = pd.DataFrame(threshold_analysis)
print("\nАнализ порогов для времени просмотра:")
print(threshold_df)

# Построение графика, демонстрирующего пороговое значение

In [ ]:
# Данные из анализа
thresholds = [5, 10, 15, 30, 60]
churn_below = [0.945, 0.891, 0.854, 0.802, 0.790]
churn_above = [0.654, 0.509, 0.405, 0.206, 0.077]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Отток по обе стороны порога
x = np.arange(len(thresholds))
width = 0.35

ax1.bar(x - width/2, churn_below, width, label='< порога', color='red', alpha=0.7)
ax1.bar(x + width/2, churn_above, width, label='≥ порога', color='green', alpha=0.7)
ax1.set_xlabel('Порог (минут в день)')
ax1.set_ylabel('Доля оттока')
ax1.set_title('Отток по разные стороны порога')
ax1.set_xticks(x)
ax1.set_xticklabels(thresholds)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Подписи значений к графику 1
for i, (cb, ca) in enumerate(zip(churn_below, churn_above)):
    ax1.text(i - width/2, cb + 0.02, f'{cb:.0%}', ha='center', fontsize=9)
    ax1.text(i + width/2, ca + 0.02, f'{ca:.0%}', ha='center', fontsize=9)

# График 2: Выигрыш от преодоления порога
ax2.bar(x, threshold_df['diff'], color='blue', alpha=0.7)
ax2.set_xlabel('Порог (минут в день)')
ax2.set_ylabel('Разница в оттоке (п.п.)')
ax2.set_title('Насколько снижается отток при преодолении порога')
ax2.set_xticks(x)
ax2.set_xticklabels(thresholds)
ax2.grid(axis='y', alpha=0.3)

# Выделение порога 30 минут
ax2.bar(3, threshold_df.loc[3, 'diff'], color='darkblue')

# Подписи значений к графику 2
for i, diff in enumerate(threshold_df['diff']):
    ax2.text(i, diff + 0.01, f'{diff:.0%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Гипотеза 2. Порог ценности в 30 минут просмотра влияет на вероятность оттока

Порог 30 мин. Пользователи кто смотрит < 30 минут уходит в 80% случаев (не видит ценности), кто более 30 минут уходит в 20% случаев (видит ценность, возможно смотрит какой-то конкретный контент).

### Стратегия проверки

Разделим пользователей на 2 группы:
1. пользователи с avg_min_watch_daily < 30 минут
2. пользователи с avg_min_watch_daily ≥ 30 минут.

Проверим, различается ли статистически доля оттока в каждой группе.

### Статистический метод

Z-тест для пропорций (основной метод) и доверительные интервалы.

#### Обоснование выбора теста:

Для долей/пропорций работает Центральная Предельная Теорема (ЦПТ):
Выборка достаточно большая (n > 30).
Распределение выборочной доли стремится к нормальному.
Не требуется нормальность исходных данных

# Проверка гипотезы статистическими тестами

### H₀: Время просмотра не влияет на решение о покупке подписки. Доли оттока равны.
### H₁: Доля оттока в группе с просмотром <30 минут БОЛЬШЕ, чем в группе с просмотром ≥30 минут.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Проверяем порог 30 минут
below_30 = df[df['avg_min_watch_daily'] < 30]
above_30 = df[df['avg_min_watch_daily'] >= 30]

counts = [below_30['churn'].sum(), above_30['churn'].sum()]
nobs = [len(below_30), len(above_30)]

z_stat, p_value = proportions_ztest(counts, nobs, alternative='larger')
print(f"\nСтатистическая проверка порога 30 минут:")
print(f"Z-статистика: {z_stat:.2f}")
print(f"p-value: {p_value:.10f}")

if p_value < 0.0001:
    print("Разница чрезвычайно статистически значима (p < 0.0001)")

# Относительный риск
rr = below_30['churn'].mean() / above_30['churn'].mean()
print(f"\nОтносительный риск: {rr:.1f}")
print(f"Риск оттока при <30 мин в {rr:.1f} раза выше")

In [ ]:
# Рассчитываем проценты
p1 = counts[0] / nobs[0]  # доля оттока в группе <30 мин
p2 = counts[1] / nobs[1]  # доля оттока в группе ≥30 мин


fig, ax2 = plt.subplots(1, 1, figsize=(6, 5))

x_pos = [0, 1]
group_names = ['<30 мин', '≥30 мин']
colors_samples = ['#FF6B6B', '#4ECDC4']
# Проценты оттока
percentages = [p1 * 100, p2 * 100]  # в процентах
colors_churn = ['#FF5252', '#2ECC71']

bars2 = ax2.bar(x_pos, percentages, color=colors_churn, alpha=0.8, edgecolor='black')

ax2.set_title('Процент оттока по группам', fontsize=14, fontweight='bold')
ax2.set_ylabel('Процент оттока (%)', fontsize=12)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(group_names, fontsize=12)
ax2.set_ylim(0, max(percentages) * 1.15)
ax2.grid(axis='y', alpha=0.3)

# Подписи над столбиками
for i, bar in enumerate(bars2):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

    # Абсолютные числа в оттоке
    ax2.text(bar.get_x() + bar.get_width()/2., height/2,
            f'{counts[i]:,} из {nobs[i]:,}', ha='center', va='center',
            fontsize=10, color='white', fontweight='bold')

# Линия сравнения и стрелка
diff = p1 - p2
ax2.plot([0, 1], [percentages[0], percentages[1]], 'k--', alpha=0.5, linewidth=1)
ax2.annotate('', xy=(1, percentages[1] + 5), xytext=(0, percentages[1] + 5),
            arrowprops=dict(arrowstyle='<->', color='black', lw=2))
ax2.text(0.5, percentages[1] + 7, f'Разница: {diff*100:.1f}%',
         ha='center', fontsize=11, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

# p-value и статистика на графике
plt.figtext(0.5, 0.01,
           f'Z-тест: z = {z_stat:.2f} | p-value = {p_value:.6f} | '
           f'Относительный риск: {p1/p2:.1f}×',
           ha='center', fontsize=12, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

# Цветовая индикация значимости
if p_value < 0.05:
    significance_box = dict(boxstyle='round', facecolor='lightgreen', alpha=0.9)
    significance_text = 'Статистически значимо (p < 0.05)'
else:
    significance_box = dict(boxstyle='round', facecolor='lightcoral', alpha=0.9)
    significance_text = 'Не значимо (p ≥ 0.05)'

plt.figtext(0.5, 0.95, significance_text,
           ha='center', fontsize=12, fontweight='bold',
           bbox=significance_box)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


# Построение доверительных интервалов

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.proportion import proportion_confint

# Данные
p_below = 0.802  # 80.2% оттока
p_above = 0.206  # 20.6% оттока
n_below, n_above = 650, 350

# Доверительные интервалы 95%
ci_below = proportion_confint(count=int(p_below*n_below), nobs=n_below, alpha=0.05)
ci_above = proportion_confint(count=int(p_above*n_above), nobs=n_above, alpha=0.05)

fig, ax = plt.subplots(figsize=(10, 4))

# Первая точка (<30 мин) - крвсная
ax.errorbar(x=0, y=p_below,
            yerr=[[p_below - ci_below[0]], [ci_below[1] - p_below]],
            fmt='o', capsize=10, capthick=2, markersize=10,
            color='red', ecolor='black', linewidth=2, label='<30 мин/день')

# Вторая точка (≥30 мин) - зеленая
ax.errorbar(x=1, y=p_above,
            yerr=[[p_above - ci_above[0]], [ci_above[1] - p_above]],
            fmt='o', capsize=10, capthick=2, markersize=10,
            color='green', ecolor='black', linewidth=2, label='≥30 мин/день')

ax.set_xlim(-0.5, 1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(['<30 мин/день', '≥30 мин/день'])
ax.set_ylabel('Доля оттока', fontsize=12)
ax.set_title('Проверка гипотезы: доверительные интервалы не пересекаются - гипотеза подтверждена',
             fontsize=14, fontweight='bold')

# Добавляем значения
ax.text(0, p_below + 0.05, f'{p_below:.1%}', ha='center', fontsize=11, fontweight='bold')
ax.text(1, p_above + 0.05, f'{p_above:.1%}', ha='center', fontsize=11, fontweight='bold')

# Линия значимости
if ci_below[1] > ci_above[0]:
    ax.axhspan(ci_below[1], ci_above[0], alpha=0.2, color='yellow',
               label='Зона не пересечения (значимо!)')
    ax.text(0.5, (ci_below[1] + ci_above[0])/2, 'НЕ ПЕРЕСЕКАЮТСЯ!\np < 0.001',
            ha='center', va='center', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### Выводы: Разница в оттоке между группами (<30 мин: 80.2%, ≥30 мин: 20.6%) является статистически высокозначимой (p < 0.0001), что позволяет с уверенностью утверждать, что время просмотра является ключевым фактором оттока.

### Рекомендации: Сделать порог 30 минут ключевой метрикой для маркетинга и аналитики. Все усилия должны быть направлены на то, чтобы пользователь как можно быстрее достиг этого уровня.

# Связь категориальных признаков и целевой переменной внутри группы с просмотрами менее 30 минут.

## Гипотеза 2.1. Среди пользователей, смотрящих менее 30 минут в день, существуют статистически значимые различия в оттоке в зависимости от категориальных признаков (город, устройство, источник трафика, любимый жанр).

### Стратегия проверки

Выделение целевой группы: фильтрация пользователей с avg_min_watch_daily < 30
Для каждого one-hot столбца (город, устройство, источник, жанр):
 - Построение таблицы сопряженности 2×2 с переменной churn
 - Расчет коэффициента фи (φ) для измерения силы и направления связи
 - Расчет p-value через тест хи-квадрат
Сортировка признаков по абсолютному значению |φ|

### Статистический метод

Тест хи-квадрат Пирсона для независимости с расчетом коэффициента фи (φ).

#### Обоснование выбора теста:

Структура данных оптимальна для метода:
- Целевая переменная churn — бинарная (0/1)
- Признаки после one-hot кодирования — бинарные (0/1)

# Проверка гипотезы статистическими тестами

### H₀: Среди пользователей, смотрящих менее 30 минут в день, категориальный признак X и отток пользователей (churn) статистически независимы. P(churn=1 | X=1, avg_min_watch_daily < 30) = P(churn=1 | X=0, avg_min_watch_daily < 30)
### H₁: Среди пользователей, смотрящих менее 30 минут в день, существует статистически значимая зависимость между категориальным признаком X и оттоком пользователей. P(churn=1 | X=1, avg_min_watch_daily < 30) ≠ P(churn=1 | X=0, avg_min_watch_daily < 30)

In [ ]:
# Фильтруем только пользователей с <30 минут просмотра
less_than_30 = data_encoded[data_encoded['avg_min_watch_daily'] < 30]

# Проверяем размер группы
print(f"Пользователей с просмотром <30 мин: {len(less_than_30)}")
print(f"Отток в этой группе: {less_than_30['churn'].mean():.1%}")

results = []

for col in less_than_30.columns:
    if any(cat in col for cat in ['city_', 'device_', 'source_', 'favourite_genre_']):
        conf_matrix = pd.crosstab(less_than_30[col], less_than_30['churn'])

        # Фи-коэффициент
        a, b, c, d = conf_matrix.values.flatten()
        phi = (a*d - b*c) / np.sqrt((a+b)*(c+d)*(a+c)*(b+d))

        # p-value
        _, p_value, _, _ = chi2_contingency(conf_matrix)

        results.append({
            'feature': col,
            'phi_coefficient': round(phi, 4),
            'p_value': round(p_value, 6),
            'significant': p_value < 0.05
        })

# Создаем и сортируем DataFrame
less_30_results = pd.DataFrame(results)
less_30_results = less_30_results.sort_values('phi_coefficient', key=abs, ascending=False)

print("Категориальные признаки для группы <30 мин:")
print(less_30_results[['feature', 'phi_coefficient', 'p_value', 'significant']])

# Дополнительно: какой признак сильнее всего влияет на отток в этой группе
top_feature = less_30_results.iloc[0]
print(f"\nСамый влияющий признак: {top_feature['feature']}")
print(f"Phi = {top_feature['phi_coefficient']} (сила связи)")
print(f"p-value = {top_feature['p_value']}")

### Выводы: Organic-пользователи немного чаще отказываются от подписки при малом просмотре. Слабая связь (φ=0.0125)

### Рекомендации: Проверить качество рекомендаций для organic-пользователей. Улучшить onboarding для organic-трафика — быстрее показывать ценный контент. Не оптимизировать другие признаки — эффекты слишком слабые.

# Связь категориальных признаков и целевой переменной внутри группы с просмотрами более 30 минут.

## Гипотеза 2.2. Среди пользователей, смотрящих более 30 минут в день, существуют статистически значимые различия в оттоке в зависимости от категориальных признаков (город, устройство, источник трафика, любимый жанр).

### Стратегия проверки

Выделение целевой группы: фильтрация пользователей с avg_min_watch_daily > 30
Для каждого one-hot столбца (город, устройство, источник, жанр):
 - Построение таблицы сопряженности 2×2 с переменной churn
 - Расчет коэффициента фи (φ) для измерения силы и направления связи
 - Расчет p-value через тест хи-квадрат
Сортировка признаков по абсолютному значению |φ|

### Статистический метод

Тест хи-квадрат Пирсона для независимости с расчетом коэффициента фи (φ).

#### Обоснование выбора теста:

Структура данных оптимальна для метода:
- Целевая переменная churn — бинарная (0/1)
- Признаки после one-hot кодирования — бинарные (0/1)

# Проверка гипотезы статистическими тестами

### H₀: Среди пользователей, смотрящих более 30 минут в день, категориальный признак X и отток пользователей (churn) статистически независимы. P(churn=1 | X=1, avg_min_watch_daily < 30) = P(churn=1 | X=0, avg_min_watch_daily < 30)
### H₁: Среди пользователей, смотрящих более 30 минут в день, существует статистически значимая зависимость между категориальным признаком X и оттоком пользователей. P(churn=1 | X=1, avg_min_watch_daily < 30) ≠ P(churn=1 | X=0, avg_min_watch_daily < 30)

In [ ]:
# Фильтруем только пользователей с >30 минут просмотра
more_than_30 = data_encoded[data_encoded['avg_min_watch_daily'] > 30]

# Проверяем размер группы
print(f"Пользователей с просмотром >30 мин: {len(more_than_30)}")
print(f"Отток в этой группе: {more_than_30['churn'].mean():.1%}")

results = []

for col in more_than_30.columns:
    if any(cat in col for cat in ['city_', 'device_', 'source_', 'favourite_genre_']):
        conf_matrix = pd.crosstab(more_than_30[col], more_than_30['churn'])

        # Фи-коэффициент
        a, b, c, d = conf_matrix.values.flatten()
        phi = (a*d - b*c) / np.sqrt((a+b)*(c+d)*(a+c)*(b+d))

        # p-value
        _, p_value, _, _ = chi2_contingency(conf_matrix)

        results.append({
            'feature': col,
            'phi_coefficient': round(phi, 4),
            'p_value': round(p_value, 6),
            'significant': p_value < 0.05
        })

# Создаем и сортируем DataFrame
more_30_results = pd.DataFrame(results)
more_30_results = more_30_results.sort_values('phi_coefficient', key=abs, ascending=False)

print("Категориальные признаки для группы >30 мин:")
print(more_30_results[['feature', 'phi_coefficient', 'p_value', 'significant']])

# Дополнительно: какой признак сильнее всего влияет на отток в этой группе
top_feature = more_30_results.iloc[0]
print(f"\nСамый влияющий признак: {top_feature['feature']}")
print(f"Phi = {top_feature['phi_coefficient']} (сила связи)")
print(f"p-value = {top_feature['p_value']}")

### Выводы: Среди долго смотрящих (>30 мин) пользователи из Екатеринбурга имеют повышенный отток. Слабая связь (φ=0.097)

### Рекомендации: Главный фокус: понять, почему даже вовлечённые пользователи из Екатеринбурга уходят чаще. Проверить качество стриминга в регионе, проанализировать доступный контент (возможно, нет локального), проанализировать ценовую чувствительность — нужны региональные промо-тарифы.

# Проанализируем изменение конверсии в зависимости от количества дней логинов в триал по группам: с просмотрами < 30 минут и > 30 минут

In [ ]:
# Создаем группы
df['watch_group'] = df['avg_min_watch_daily'] >= 30
group_names = {True: '≥30 мин', False: '<30 мин'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (group_val, group_name) in enumerate(group_names.items()):
    group_data = df[df['watch_group'] == group_val]

    # Расчет конверсии
    conversion_by_days = (
        group_data.groupby("number_of_days_logged")
                  .apply(lambda x: 1 - x["churn"].mean())
                  .reset_index(name="conversion_rate")
    )

    # График
    axes[idx].plot(
        conversion_by_days["number_of_days_logged"],
        conversion_by_days["conversion_rate"],
        marker="o", linewidth=2, markersize=8
    )

    axes[idx].set_xlabel("Количество дней логинов в триал")
    axes[idx].set_ylabel("Конверсия в подписку")
    axes[idx].set_title(f"Группа: {group_name} (n={len(group_data)})")
    axes[idx].grid(True)
    axes[idx].set_ylim(0, 1)

    # Добавляем аннотации для ключевых точек
    max_days = conversion_by_days["number_of_days_logged"].max()
    if len(conversion_by_days) > 0:
        initial_rate = conversion_by_days.iloc[0]["conversion_rate"]
        final_rate = conversion_by_days.iloc[-1]["conversion_rate"]

        axes[idx].annotate(f"Начало: {initial_rate:.1%}",
                          xy=(conversion_by_days.iloc[0]["number_of_days_logged"], initial_rate),
                          xytext=(5, 10), textcoords='offset points')

        axes[idx].annotate(f"Конец: {final_rate:.1%}",
                          xy=(max_days, final_rate),
                          xytext=(-40, 10), textcoords='offset points')

plt.tight_layout()
plt.show()

# Статистика по группам
print("Сводная статистика:")
for group_val, group_name in group_names.items():
    group_data = df[df['watch_group'] == group_val]
    avg_conversion = 1 - group_data['churn'].mean()
    print(f"\n{group_name}:")
    print(f"  Количество пользователей: {len(group_data)}")
    print(f"  Средняя конверсия: {avg_conversion:.1%}")
    print(f"  Среднее дней логинов: {group_data['number_of_days_logged'].mean():.1f}")

### Вывод:  3-й день — критический момент. Нужно удержать активность пользователей среди пользователей смотрящих > 30 минут.

# Связь категориальных признаков и целевой переменной внутри группы с просмотрами более 30 минут и 3 днями логирований

## Гипотеза 2.3. Среди пользователей, активных в течении 3х дней и смотрящих более 30 минут в день, существуют статистически значимые различия в оттоке в зависимости от категориальных признаков (город, устройство, источник трафика, любимый жанр).

### Стратегия проверки

Выделение целевой группы: фильтрация пользователей с avg_min_watch_daily > 30 и number_of_days_logged = 3
Для каждого one-hot столбца (город, устройство, источник, жанр):
 - Построение таблицы сопряженности 2×2 с переменной churn
 - Расчет коэффициента фи (φ) для измерения силы и направления связи
 - Расчет p-value через тест хи-квадрат
Сортировка признаков по абсолютному значению |φ|

### Статистический метод

Тест хи-квадрат Пирсона для независимости с расчетом коэффициента фи (φ).

#### Обоснование выбора теста:

Структура данных оптимальна для метода:
- Целевая переменная churn — бинарная (0/1)
- Признаки после one-hot кодирования — бинарные (0/1)

# Проверка гипотезы статистическими тестами

### H₀: Среди пользователей, активных в течении 3х дней и смотрящих более 30 минут в день, категориальный признак X и отток пользователей (churn) статистически независимы. P(churn=1 | X=1, avg_min_watch_daily > 30, number_of_days_logged = 3) = P(churn=1 | X=0, avg_min_watch_daily > 30, number_of_days_logged = 3)
### H₁: Среди пользователей, активных в течении 3х дней и смотрящих более 30 минут в день, существует статистически значимая зависимость между категориальным признаком X и оттоком пользователей. P(churn=1 | X=1, avg_min_watch_daily > 30, number_of_days_logged = 3) ≠ P(churn=1 | X=0, avg_min_watch_daily > 30, number_of_days_logged = 3)

In [ ]:
# Фильтруем пользователей с >30 минут просмотра и 3 днями логинов
filtered_group = data_encoded[
    (data_encoded['avg_min_watch_daily'] > 30) &
    (data_encoded['number_of_days_logged'] == 3)
]

# Проверяем размер группы
print(f"Пользователей с просмотром >30 мин и 3 днями логинов: {len(filtered_group)}")
print(f"Отток в этой группе: {filtered_group['churn'].mean():.1%}")

results = []

for col in filtered_group.columns:
    if any(cat in col for cat in ['city_', 'device_', 'source_', 'favourite_genre_']):
        conf_matrix = pd.crosstab(filtered_group[col], filtered_group['churn'])

        # Фи-коэффициент
        a, b, c, d = conf_matrix.values.flatten()
        phi = (a*d - b*c) / np.sqrt((a+b)*(c+d)*(a+c)*(b+d))

        # p-value
        _, p_value, _, _ = chi2_contingency(conf_matrix)

        results.append({
            'feature': col,
            'phi_coefficient': round(phi, 4),
            'p_value': round(p_value, 6),
            'significant': p_value < 0.05
        })

# Создаем и сортируем DataFrame
filtered_results = pd.DataFrame(results)
filtered_results = filtered_results.sort_values('phi_coefficient', key=abs, ascending=False)

print("\nКатегориальные признаки для группы (>30 мин, 3 дня логинов):")
print(filtered_results[['feature', 'phi_coefficient', 'p_value', 'significant']])

### Выводы: Среди пользователей, долго смотрящих (>30 мин) и имеющий 3 дня логирований: из малых городов имеют повышенный отток. Также performance-пользователи имеют повышенный отток среди этой категории. (32.2% vs 20.6%).

### Рекомендации: Для пользователей из малых городов (city_Other) продумать персональные промо-акции на 3-й день. Сделать опрос о причинах недовольства. Для performance-трафика проверить соответствие рекламы и реального контента. Для всех в этой группе: отправить Push-уведомление на 3-й день с подборкой контента и напомнить о преимуществах подписки.

# Проанализируем как конверсия меняется в зависимости от количества дней логинов в триал

In [ ]:
conversion_by_days = (
    df.groupby("number_of_days_logged")
      .apply(lambda x: 1 - x["churn"].mean()) 
      .reset_index(name="conversion_rate")
)

conversion_by_days

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(
    conversion_by_days["number_of_days_logged"],
    conversion_by_days["conversion_rate"],
    marker="o"
)

plt.xlabel("Количество дней логинов в триал")
plt.ylabel("Конверсия в подписку")
plt.title("Зависимость конверсии от количества дней логинов")
plt.grid(True)
plt.show()


Наблюдается **спад конверсии на 5 и 7 дни**. Люди теряют интерес или пользуются только пробным периодом, а затем уходят

## Гипотеза 3: Конверсия падает на 5-й день триала

Спад конверсии на 5-й день триала (20.4% против 22.5% на 4-й день) обусловлен тем, что пользователи, которые не нашли достаточно ценного контента к середине триала, снижают активность просмотра, что и приводит к снижению конверсии.

### Стратегия проверки

Разделим пользователей на 2 группы:
1. те, которые были активны на 5-й день и совершили подписку
2. те, которые были активны на 5-й день и не совершили подписку.

Чтобы проверить, предсказывает ли активность перед спадом сам спад конверсии, проанализируем среднее время просмотра на 4-й день триала (день перед спадом).

### Статистический метод

Тест Манна-Уитни

**Идея:** Пользователи с 5 днями активности, но низким средним временем просмотра в день, демонстрируют поверхностное использование сервиса, что приводит к снижению конверсии.

In [ ]:
import scipy.stats as stats

# Рассчитаем конверсию по дням активности
conversion_by_days = (
    df.groupby("number_of_days_logged")
      .agg(
          total_users=('user_id', 'count'),
          conversions=('churn', lambda x: (x == 0).sum()),
          conversion_rate=('churn', lambda x: (1 - x.mean()) * 100),
          avg_watch_time=('avg_min_watch_daily', 'mean')
      )
      .reset_index()
)

print("Конверсия и среднее время просмотра по дням активности:")
print(conversion_by_days.to_string())


# Сравним пользователей с 4 и 5 днями активности
users_4_days = df[df['number_of_days_logged'] == 4].copy()
users_5_days = df[df['number_of_days_logged'] == 5].copy()

print(f"\nПользователей с 4 днями активности: {len(users_4_days)}")
print(f"Пользователей с 5 днями активности: {len(users_5_days)}")

# Сравнение среднего времени просмотра между группами 4 и 5 дней
print(f"\nСравнение среднего времени просмотра:")
print(f"4 дня: {users_4_days['avg_min_watch_daily'].mean():.1f} мин/день")
print(f"5 дней: {users_5_days['avg_min_watch_daily'].mean():.1f} мин/день")

# Тест Манна-Уитни для сравнения распределений
stat_4_5, p_4_5 = stats.mannwhitneyu(
    users_4_days['avg_min_watch_daily'].dropna(),
    users_5_days['avg_min_watch_daily'].dropna(),
    alternative='two-sided'
)

print(f"\nТест Манна-Уитни (4 дня vs 5 дней):")
print(f"p-value = {p_4_5:.5f}")
if p_4_5 < 0.05:
    print("Есть статистически значимая разница в среднем времени просмотра")
else:
    print("Нет статистически значимой разницы в среднем времени просмотра")

# Основной тест: сравнение вовлеченности внутри группы 5 дней
# Разделим пользователей с 5 днями активности на конвертированных и неконвертированных
users_5_converted = users_5_days[users_5_days['churn'] == 0]
users_5_not_converted = users_5_days[users_5_days['churn'] == 1]

print(f"\nАнализ внутри группы с 5 днями активности:")
print(f"Конвертировалось: {len(users_5_converted)} ({len(users_5_converted)/len(users_5_days)*100:.1f}%)")
print(f"Не конвертировалось: {len(users_5_not_converted)} ({len(users_5_not_converted)/len(users_5_days)*100:.1f}%)")
print(f"\nСреднее время просмотра:")
print(f"Конвертированные: {users_5_converted['avg_min_watch_daily'].mean():.1f} мин/день")
print(f"Неконвертированные: {users_5_not_converted['avg_min_watch_daily'].mean():.1f} мин/день")

# Тест Манна-Уитни для сравнения внутри группы 5 дней
stat_5_groups, p_5_groups = stats.mannwhitneyu(
    users_5_converted['avg_min_watch_daily'].dropna(),
    users_5_not_converted['avg_min_watch_daily'].dropna(),
    alternative='greater'  # Проверяем, что у конвертированных время ВЫШЕ
)

print(f"\nТест Манна-Уитни (внутри 5 дней, конвертированные vs неконвертированные):")
print(f"p-value = {p_5_groups:.5f}")
print(f"U-статистика = {stat_5_groups:.0f}")

if p_5_groups < 0.05:
    print("У конвертированных пользователей с 5 днями активности")
    print("Среднее время просмотра статистически значимо выше")
else:
    print("Нет статистически значимой разницы во времени просмотра")
    print(" Между конвертированными и неконвертированными пользователями с 5 днями активности")

# 5. Дополнительный анализ: сравним с пользователями, у которых 4 дня активности
users_4_converted = users_4_days[users_4_days['churn'] == 0]
users_4_not_converted = users_4_days[users_4_days['churn'] == 1]

print(f"\nСравнение с пользователями 4 дней активности:")
print(f"Конвертированные (4 дня): {users_4_converted['avg_min_watch_daily'].mean():.1f} мин/день")
print(f"Конвертированные (5 дней): {users_5_converted['avg_min_watch_daily'].mean():.1f} мин/день")
print(f"Разница: {users_5_converted['avg_min_watch_daily'].mean() - users_4_converted['avg_min_watch_daily'].mean():.1f} мин/день")

In [ ]:
# Создадим комплексную визуализацию
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Анализ гипотезы: "Низкая ежедневная вовлеченность объясняет спад конверсии на 5-й день активности"',
             fontsize=14, fontweight='bold', y=1.02)

# График 1: Конверсия по дням активности
ax1 = axes[0, 0]
bars = ax1.bar(conversion_by_days['number_of_days_logged'],
               conversion_by_days['conversion_rate'],
               color=['skyblue' if x != 5 else 'coral' for x in conversion_by_days['number_of_days_logged']],
               alpha=0.7)
ax1.set_xlabel("Количество дней с активностью", fontsize=11)
ax1.set_ylabel("Конверсия, %", fontsize=11)
ax1.set_title("Конверсия по дням активности", fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Добавляем значения на столбцы
for bar, conv in zip(bars, conversion_by_days['conversion_rate']):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{conv:.1f}%', ha='center', va='bottom', fontsize=10)

# График 2: Среднее время просмотра по дням активности
ax2 = axes[0, 1]
ax2.bar(conversion_by_days['number_of_days_logged'],
        conversion_by_days['avg_watch_time'],
        color=['lightgreen' if x != 5 else 'gold' for x in conversion_by_days['number_of_days_logged']],
        alpha=0.7)
ax2.set_xlabel("Количество дней с активностью", fontsize=11)
ax2.set_ylabel("Среднее время просмотра, мин/день", fontsize=11)
ax2.set_title("Средняя ежедневная вовлеченность", fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# График 3: Сравнение распределений времени просмотра (4 vs 5 дней)
ax3 = axes[0, 2]
# Боксплот для сравнения распределений
box_data = [users_4_days['avg_min_watch_daily'].dropna(),
            users_5_days['avg_min_watch_daily'].dropna()]
labels = ['4 дня активности', '5 дней активности']
boxplot = ax3.boxplot(box_data, labels=labels, patch_artist=True, widths=0.6)

# Раскрашиваем боксы
colors_box = ['lightblue', 'peachpuff']
for patch, color in zip(boxplot['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax3.set_ylabel("Время просмотра, мин/день", fontsize=11)
ax3.set_title(f"Сравнение распределений\n(p={p_4_5:.4f})", fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# График 4: Сравнение внутри группы 5 дней
ax4 = axes[1, 0]

box_data_5 = [users_5_converted['avg_min_watch_daily'].dropna(),
              users_5_not_converted['avg_min_watch_daily'].dropna()]
labels_5 = ['Конвертированные\n(5 дней)', 'Неконвертированные\n(5 дней)']
boxplot_5 = ax4.boxplot(box_data_5, labels=labels_5, patch_artist=True, widths=0.6)

# Раскрашиваем
colors_5 = ['lightgreen', 'lightcoral']
for patch, color in zip(boxplot_5['boxes'], colors_5):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Добавляем средние значения
for i, data in enumerate(box_data_5, 1):
    mean_val = np.mean(data)
    ax4.scatter(i, mean_val, color='black', zorder=3, s=60)

ax4.set_ylabel("Время просмотра, мин/день", fontsize=11)
ax4.set_title(f"Сравнение внутри 5 дней\n(p={p_5_groups:.4f})", fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# График 5: Зависимость конверсии от времени просмотра для 5 дней
ax5 = axes[1, 1]
# Создаем бины по времени просмотра
users_5_days['watch_bin'] = pd.qcut(users_5_days['avg_min_watch_daily'],
                                     q=4,
                                     labels=['Q1 (низкая)', 'Q2', 'Q3', 'Q4 (высокая)'])
bin_conversion = users_5_days.groupby('watch_bin')['churn'].mean().apply(lambda x: (1-x)*100)

colors_bin = ['lightcoral', 'peachpuff', 'lightblue', 'lightgreen']
bars_bin = ax5.bar(range(len(bin_conversion)), bin_conversion, color=colors_bin, alpha=0.8)

ax5.set_xlabel("Квартиль ежедневной вовлеченности", fontsize=11)
ax5.set_ylabel("Конверсия, %", fontsize=11)
ax5.set_title("Конверсия по уровню вовлеченности\n(только 5 дней активности)",
              fontsize=12, fontweight='bold')
ax5.set_xticks(range(len(bin_conversion)))
ax5.set_xticklabels(bin_conversion.index, rotation=45, ha='right')
ax5.grid(True, alpha=0.3, axis='y')

# Добавляем значения
for bar, conv in zip(bars_bin, bin_conversion):
    height = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{conv:.1f}%', ha='center', va='bottom', fontsize=10)

# График 6: Совместная визуализация конверсии и вовлеченности
ax6 = axes[1, 2]

# Создаем бины вовлеченности на основе времени просмотра
engagement_bins_series = pd.cut(users_5_days['avg_min_watch_daily'], bins=10)

# Группируем по этим бинам - используем engagement_bins_series вместо имени столбца
grouped = users_5_days.groupby(engagement_bins_series).agg(
    conversion_rate=('churn', lambda x: (1 - x.mean()) * 100),
    avg_engagement=('avg_min_watch_daily', 'mean'),
    count=('user_id', 'count')
).reset_index()

# Переименуем индексный столбец для удобства
grouped = grouped.rename(columns={'avg_min_watch_daily': 'engagement_bins'})

# Отфильтруем бины с малым количеством наблюдений
grouped = grouped[grouped['count'] > 10]

# Двойная ось
ax6_bar = ax6.bar(range(len(grouped)), grouped['count'],
                  alpha=0.5, color='gray', label='Количество пользователей')
ax6.set_xlabel("Бины среднего времени просмотра (мин/день)", fontsize=11)
ax6.set_ylabel("Количество пользователей", fontsize=11, color='gray')
ax6.tick_params(axis='y', labelcolor='gray')

ax6_line = ax6.twinx()
ax6_line.plot(range(len(grouped)), grouped['conversion_rate'],
              color='crimson', marker='o', linewidth=2, markersize=6,
              label='Конверсия, %')
ax6_line.set_ylabel("Конверсия, %", fontsize=11, color='crimson')
ax6_line.tick_params(axis='y', labelcolor='crimson')
ax6_line.set_ylim(0, 100)

# Устанавливаем подписи оси X
ax6.set_xticks(range(len(grouped)))
ax6.set_xticklabels([f"{interval.left:.0f}-{interval.right:.0f}"
                     for interval in grouped['engagement_bins']],
                    rotation=45, ha='right')

ax6.set_title("Зависимость конверсии от вовлеченности", fontsize=12, fontweight='bold')


# Поиск оптимального порога вовлеченности
print("\nАнализ оптимального порога вовлеченности\n")

# Создаем различные пороги и смотрим конверсию
thresholds = [10, 15, 20, 25, 30, 40, 50]
results = []

for threshold in thresholds:
    above_threshold = users_5_days[users_5_days['avg_min_watch_daily'] >= threshold]
    below_threshold = users_5_days[users_5_days['avg_min_watch_daily'] < threshold]

    if len(above_threshold) > 10 and len(below_threshold) > 10:
        conv_above = (1 - above_threshold['churn'].mean()) * 100
        conv_below = (1 - below_threshold['churn'].mean()) * 100

        results.append({
            'threshold': threshold,
            'users_above': len(above_threshold),
            'users_below': len(below_threshold),
            'conv_above': conv_above,
            'conv_below': conv_below,
            'diff': conv_above - conv_below
        })

results_df = pd.DataFrame(results)
print("\nКонверсия при разных порогах вовлеченности (для 5 дней активности):")
print(results_df.to_string())

# Находим оптимальный порог
if not results_df.empty:
    optimal = results_df.loc[results_df['diff'].idxmax()]
    print(f"\nОптимальный порог вовлеченности: {optimal['threshold']} мин/день")
    print(f"   Конверсия выше порога: {optimal['conv_above']:.1f}%")
    print(f"   Конверсия ниже порога: {optimal['conv_below']:.1f}%")
    print(f"   Разница: +{optimal['diff']:.1f}%")

# Выводы
if p_5_groups < 0.05:
    print("\nГипотеза подтверждена\n")

Проблема не в количестве дней активности, а в качестве использования дней.

Пользователи с 5 днями активности делятся на две четкие группы:

* Глубоко вовлеченные (14+ мин/день) → конвертируются почти так же хорошо, как и те, у кого 4 дня
* Поверхностно вовлеченные (~6 мин/день) → редко конвертируются

Проведем дополнительный анализ: почему именно 5 дней?

In [ ]:
# 1. Анализ распределения
print("\n1. Расределение пользователей с 5 днями активности на оставшихся и ушедших:")
print(f"Всего пользователей с 5 днями активности: {len(users_5_days)}")
print(f"Из них подписались: {len(users_5_converted)} ({len(users_5_converted)/len(users_5_days)*100:.1f}%)")
print(f"Из них не подписались: {len(users_5_not_converted)} ({len(users_5_not_converted)/len(users_5_days)*100:.1f}%)")

# 2. Анализ порога вовлеченности
print("\n2. Критический порог вовлеченности:")

# Рассчитаем медиану среднего минимального просмотра для не подписавшихся
median_not_converted = users_5_not_converted['avg_min_watch_daily'].median()
print(f"Медианная вовлеченность неконвертированных: {median_not_converted:.1f} мин/день")

# Сколько не подписавшихся с медианой выше медианы под?
converted_median = users_5_converted['avg_min_watch_daily'].median()
not_converted_above_threshold = users_5_not_converted[users_5_not_converted['avg_min_watch_daily'] >= converted_median]
print(f"Неконвертированных с вовлеченностью ≥{converted_median:.0f} мин: {len(not_converted_above_threshold)}")
print(f"Из них могли бы конвертироваться: ~{len(not_converted_above_threshold)/len(users_5_not_converted)*100:.1f}%")

# 3. Сравнение с другими днями активности
print("\n3. СРАВНЕНИЕ ПО ДНЯМ АКТИВНОСТИ:")
for days in [3, 4, 5, 6, 7]:
    users_days = df[df['number_of_days_logged'] == days]
    if len(users_days) > 0:
        conv_rate = (1 - users_days['churn'].mean()) * 100
        avg_watch = users_days['avg_min_watch_daily'].mean()
        conv_watch = users_days[users_days['churn'] == 0]['avg_min_watch_daily'].mean()
        not_conv_watch = users_days[users_days['churn'] == 1]['avg_min_watch_daily'].mean()

        print(f"{days} дней: Конверсия={conv_rate:.1f}%, "
              f"Среднее={avg_watch:.1f} мин, "
              f"Конверт={conv_watch:.1f} мин, "
              f"Не конверт={not_conv_watch:.1f} мин")

# 4. Анализ "потерянного потенциала"
print("\n4. ПОТЕРЯННЫЙ ПОТЕНЦИАЛ КОНВЕРСИИ:")
# Если бы неконвертированные с 5 днями имели такую же конверсию, как конвертированные с 5 дней
potential_conversions = len(users_5_not_converted) * 0.204  # 20.4% конверсии
actual_conversions = len(users_5_converted)
lost_potential = potential_conversions - actual_conversions

print(f"Фактически конвертировалось: {actual_conversions}")
print(f"Могло бы конвертироваться (при 20.4%): {potential_conversions:.0f}")
print(f"Потерянный потенциал: {lost_potential:.0f} пользователей")
print(f"Увеличение конверсии могло бы быть: +{lost_potential/len(users_5_days)*100:.1f}%")

# Создаем финальную визуализацию
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Ключевой инсайт: Качество вовлеченности важнее количества дней активности',
             fontsize=16, fontweight='bold', y=1.05)

# График 1: Две разные группы внутри 5 дней
ax1 = axes[0]
# Гистограмма распределения
ax1.hist(users_5_converted['avg_min_watch_daily'], bins=30, alpha=0.7,
         color='green', label='Конвертированные', density=True)
ax1.hist(users_5_not_converted['avg_min_watch_daily'], bins=30, alpha=0.7,
         color='red', label='Неконвертированные', density=True)
ax1.axvline(x=10, color='black', linestyle='--', linewidth=1.5,
           label='Примерный порог (10 мин)')
ax1.set_xlabel('Среднее время просмотра, мин/день', fontsize=12)
ax1.set_ylabel('Плотность распределения', fontsize=12)
ax1.set_title('Два разных типа пользователей с 5 днями активности', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Добавляем аннотации
ax1.annotate(f'Конвертированные\n14.3 мин/день',
             xy=(14.3, 0.03), xytext=(20, 0.04),
             arrowprops=dict(arrowstyle='->', color='green'),
             fontsize=10, color='green')
ax1.annotate(f'Неконвертированные\n5.8 мин/день',
             xy=(5.8, 0.05), xytext=(2, 0.06),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=10, color='red')

# График 2: Сравнение конверсии по уровню вовлеченности
ax2 = axes[1]
# Создаем более детальные бины
users_5_days['engagement_group'] = pd.cut(users_5_days['avg_min_watch_daily'],
                                          bins=[0, 5, 10, 15, 20, 30, 100],
                                          labels=['0-5', '5-10', '10-15', '15-20', '20-30', '30+'])
group_stats = users_5_days.groupby('engagement_group').agg(
    conversion_rate=('churn', lambda x: (1 - x.mean()) * 100),
    user_count=('user_id', 'count')
).reset_index()

# Убираем группы с малым количеством пользователей
group_stats = group_stats[group_stats['user_count'] > 10]

bars = ax2.bar(range(len(group_stats)), group_stats['conversion_rate'],
               color=['red', 'orange', 'yellow', 'lightgreen', 'green', 'darkgreen'])
ax2.set_xlabel('Уровень вовлеченности (мин/день)', fontsize=12)
ax2.set_ylabel('Конверсия, %', fontsize=12)
ax2.set_title('Конверсия резко растет после 10 минут в день', fontsize=14)
ax2.set_xticks(range(len(group_stats)))
ax2.set_xticklabels(group_stats['engagement_group'], rotation=45)
ax2.grid(True, alpha=0.3, axis='y')

# Добавляем значения
for i, (bar, conv, count) in enumerate(zip(bars, group_stats['conversion_rate'], group_stats['user_count'])):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 2,
             f'{conv:.0f}%\nn={count}', ha='center', va='bottom', fontsize=9)

# График 3: Сравнение всех групп
ax3 = axes[2]
# Подготовка данных для группированного боксплота
data_to_plot = []
labels = []

for days in [4, 5]:
    for churn_status in [0, 1]:
        subset = df[(df['number_of_days_logged'] == days) & (df['churn'] == churn_status)]
        if len(subset) > 10:
            data_to_plot.append(subset['avg_min_watch_daily'].dropna())
            status_name = 'Конверт' if churn_status == 0 else 'Не конверт'
            labels.append(f'{days} дней\n{status_name}')

positions = range(1, len(data_to_plot) + 1)
boxplot = ax3.boxplot(data_to_plot, positions=positions, widths=0.6, patch_artist=True)

# Раскрашиваем
colors = ['lightgreen', 'lightcoral', 'lightblue', 'pink']
for patch, color in zip(boxplot['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax3.set_xlabel('Группа пользователей', fontsize=12)
ax3.set_ylabel('Среднее время просмотра, мин/день', fontsize=12)
ax3.set_title('Сравнение вовлеченности по группам', fontsize=14)
ax3.set_xticks(positions)
ax3.set_xticklabels(labels, rotation=0)
ax3.grid(True, alpha=0.3, axis='y')

# Добавляем линии для сравнения
ax3.axhline(y=10, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax3.text(4.5, 10.5, 'Порог 10 мин/день', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

Рекомендации:

1. Идентификация рисковых пользователей

*   Определять пользователей с вовлеченностью <10 мин/день после 3-го дня
*   Особое внимание к пользователям с 5+ днями активности, но низкой вовлеченностью

2. Метрики для мониторинга

*   Среднее время просмотра в день (более важно, чем количество дней)
*   Пороговые значения: 10+ мин/день (высокая вероятность конверси) и 5-8 мин/день (высокая вероятность ухода)

3. Статегия увеличеия конверсии

Для пользователей 5+ дней с низкой вовлеченностью (<10 мин):
 - Персонализированные рекомендации контента
 - email с популярным/новым контентом
 - Предложение бесплатного продления триала на 1-2 дня
 - Онбординг-напоминания о ключевых функциях сервиса

# **Вывод**

Гипотеза частично подтверждена, но с уточнением:

Исходная гипотеза: "Спад конверсии на 5-й день обусловлен тем, что пользователи
не нашли достаточно ценного контента к середине триала"

Фактически обнаружено:
1. Количество дней активности (4 или 5) само по себе не определяет конверсию
2. Ключевой фактор - **качество** использования (среднее время просмотра в день)
3. Пользователи с 5 днями активности делятся на две группы:
   * Глубоко вовлеченные (14+ мин/день) → конвертируются хорошо (как 4-дневные)
   * Поверхностно вовлеченные (~6 мин/день) → конвертируются плохо

Исправленная гипотеза:

Снижение конверсии у пользователей с 5 днями активности объясняется не количеством  дней использования, а низкой средней ежедневной вовлеченностью. Пользователи, которые используют сервис поверхностно (менее 10 мин/день), даже при регулярном посещении, не видят достаточной ценности для конверсии.

Рекомендуемые действия:
1. Сфокусироваться на увеличении глубины вовлеченности, а не частоты посещений
2. Внедрить систему ранних предупреждений для пользователей с низкой ежедневной вовлеченностью
3. Оптимизировать качество рекомендаций и подбор интересов пользователей в первые 3-4 дня триала

# Проанализируем связь географии (городов), времени просмотров пользователей и отток

In [ ]:
# График зависимости город х время просмотра х отток
city_watch_interaction = df.groupby('city').agg({
    'churn': 'mean',
    'avg_min_watch_daily': 'mean',
    'user_id': 'count'
}).rename(columns={'user_id': 'count', 'churn': 'churn_rate'})

city_watch_interaction['churn_rate_percent'] = city_watch_interaction['churn_rate'] * 100
city_watch_interaction = city_watch_interaction.sort_values('churn_rate_percent')

plt.figure(figsize=(14, 10))

scatter = plt.scatter(city_watch_interaction['avg_min_watch_daily'],
                            city_watch_interaction['churn_rate_percent'],
                            s=city_watch_interaction['count']*2,
                            alpha=0.7,
                            c=range(len(city_watch_interaction)),
                            cmap='viridis')

for city, row in city_watch_interaction.iterrows():
   plt.annotate(city,
                       xy=(row['avg_min_watch_daily'], row['churn_rate_percent']),
                       xytext=(5, 5), textcoords='offset points',
                       fontsize=9, alpha=0.8)

plt.xlabel('Среднее время просмотра (мин/день)')
plt.ylabel('Отток, %')
plt.title('Взаимодействие: город × время просмотра × отток')
plt.grid(True, alpha=0.3)

cbar = plt.colorbar(scatter)
cbar.set_label('Индекс города', rotation=270, labelpad=15)

## Гипотеза 4: Географический фактор влияет на отток

Географический фактор (город) является значимым предиктором оттока. Пользователи в Екатеринбурге и Уфе демонстрируют паттерны поведения, отличные от других регионов: меньшее время просмотра при более высоком оттоке.

### Стратегия проверки

Разделим пользователей на 2 группы:
1. из Екатеринбурга и Уфы (проблемные города)
2. из остальных городов

### Статистический метод

Сравнение времени просмотра между группами - тест Манна-Уитни, оттока - Z-тест для пропорций.

In [ ]:
city_watch_interaction

# Сравним время просмотра у групп пользователей

### H₀: Среднее время просмотра одинаково в обеих группах

### H₁: Время просмотра меньше у пользователей из группы A

In [ ]:
from math import erf

# Функция CDF для нормального распределения
def normal_cdf(x):
    """
    Вычисляет значение функции распределения (CDF)
    стандартного нормального распределения N(0, 1).

    Использует формулу через функцию ошибок erf:
        CDF(x) = 1/2 * (1 + erf(x / sqrt(2)))

    Args:
        x (float): точка, в которой вычисляется значение CDF.

    Returns:
        float: вероятность P(Z ≤ x) для стандартного нормального распределения.
    """
    return (1 + erf(x / np.sqrt(2))) / 2

# Определим группы пользователей по городам
group_A = df[df['city'].isin(['Ufa', 'Yekaterinburg'])]['avg_min_watch_daily']
group_B = df[~df['city'].isin(['Ufa', 'Yekaterinburg'])]['avg_min_watch_daily']

# Объединяем значения в единый массив
combined = pd.concat([group_A, group_B])
ranks = combined.rank(method='average')

# Присваиваем ранги каждой группе
ranks_A = ranks.loc[group_A.index]
ranks_B = ranks.loc[group_B.index]

# Считаем статистику U
n1 = len(group_A)
n2 = len(group_B)

U1 = n1*n2 + (n1*(n1+1))/2 - ranks_A.sum()
U2 = n1*n2 - U1

U = min(U1, U2)

# Нормируем U и вычисляем z-статистику
mean_U = n1*n2 / 2
std_U = np.sqrt(n1*n2*(n1+n2+1) / 12)

z = (U - mean_U) / std_U

# Односторонний тест
p_value = 1 - normal_cdf(abs(z))

# Вывод результата
print("U-статистика: ", U)
print("Z-значение: ", z)
print("p-value: ", p_value)

- z = -1.84 < 0 -> группа A смотрит меньше, чем группа B -> H1 подтверждается
- p-value = 0.03 < 0.05 -> отвергаем H0, принимаем H1


In [ ]:
# Собираем данные в датафрейм для графика
plot_df = pd.DataFrame({
    'avg_min_watch_daily': pd.concat([group_A, group_B], ignore_index=True),
    'city_group': (['Ufa + Yekaterinburg'] * len(group_A)) + (['Other cities'] * len(group_B))
})

# Скрипичная диаграмма (violin plot)
plt.figure(figsize=(9, 5))

data = [
    plot_df.loc[plot_df['city_group'] == 'Ufa + Yekaterinburg', 'avg_min_watch_daily'].dropna().values,
    plot_df.loc[plot_df['city_group'] == 'Other cities', 'avg_min_watch_daily'].dropna().values
]

parts = plt.violinplot(data, showmeans=False, showmedians=True, showextrema=True)

plt.xticks([1, 2], ['Уфа и Екатеринбург', 'Другие города'])
plt.title('Распределение avg_min_watch_daily по группам городов')
plt.ylabel('Среднее время просмотра (мин/день)')
plt.grid(alpha=0.3)
plt.show()

Распределение среднего времени просмотра у пользователей из Екатеринбурга и Уфы смещено в сторону меньших значений по сравнению с остальными городами.

### Выводы:

Пользователи из Уфы и Екатеринбурга значительно меньше смотрят контент, чем пользователи из остальных городов.
Следовательно, географический фактор влияет на вовлечённость, а сниженное время просмотра может быть одной из причин повышенного оттока в этих городах.

## Сравним долю оттока у пользователей из групп городов
### H₀: доля оттока одинаковая в обеих группах

### H₁: доля оттока выше в группе А

In [ ]:
# Определим группы пользователей по городам
group_A = df[df['city'].isin(['Ufa', 'Yekaterinburg'])]
group_B = df[~df['city'].isin(['Ufa', 'Yekaterinburg'])]

n1 = len(group_A)
n2 = len(group_B)

k1 = group_A['churn'].sum()
k2 = group_B['churn'].sum()

p1 = k1 / n1
p2 = k2 / n2

p_pooled = (k1 + k2) / (n1 + n2)

SE = (p_pooled * (1 - p_pooled) * (1/n1 + 1/n2)) ** 0.5

z = (p1 - p2) / SE

# Односторонний тест: H1: p1 > p2
p_value = 1 - normal_cdf(z)

# Вывод результата
print("Z-значение: ", z)
print("p-value: ", p_value)

- z = 2.38 > 0 -> отток в группе А выше, чем в остальных, подтверждается H1
- p-value = 0.008 < 0.05 -> отвергаем H0, принимаем H1

In [ ]:
# Подготовка данных для визуализации
churn_rates = pd.DataFrame({
    'Группа': ['Уфа и Екатеринбург', 'Другие города'],
    'Доля оттока': [p1, p2]
})

# Построение графика
plt.figure(figsize=(6, 4))
plt.bar(churn_rates['Группа'], churn_rates['Доля оттока'])
plt.title('Доля оттока по группам городов')
plt.ylabel('Доля пользователей с оттоком')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)

# Подписи значений
for i, v in enumerate(churn_rates['Доля оттока']):
    plt.text(i, v + 0.01, f'{v:.2%}', ha='center')

plt.show()

Если объединить пользователей из Екатеринбурга и Уфы в одну группу, то доля оттока в этой группе выше, чем в агрегированной группе всех остальных городов.

### Выводы:
 Из Уфы и Екатеринбурга отток пользователей после пробной подписки выше, чем в других городах. Это подтверждает гипотезу о географическом влиянии на вовлечённость и риск оттока.
